# Football Score Prediction Notebook

This notebook builds a full scoreline forecasting pipeline for international men's matches from 2018 onward.

The workflow is:
- Fit a weighted Dixon-Coles model to estimate team attack and defense strength.
- Train a Bayesian Poisson model with MCMC using those strengths as features.
- Train an XGBoost Poisson model on the same feature set.
- Predict the fixture Argentina vs Austria and visualize score, win/draw, and top-score probabilities.
- Evaluate both models on a chronological holdout set.

In [ ]:
# Set match fixture details using either FIFA codes or dataset team names.
FIXTURE_HOME_TEAM_INPUT: str = "BEL"
FIXTURE_AWAY_TEAM_INPUT: str = "IRN"
FIXTURE_NEUTRAL: bool = False

# String constants
PROBABILITY_AXIS_LABEL: str = "Probability (%)"

In [ ]:
# Set Root
import sys
from pathlib import Path

repo_root = Path.cwd()
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / "common" / "fifa_country_codes.py").exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from io import BytesIO
from typing import Dict, Iterable, Tuple
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import gammaln, logsumexp
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
from xgboost import XGBRegressor

from common.fifa_country_codes import FIFA_TO_DATASET_TEAM

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")

RNG: np.random.Generator = np.random.default_rng(42)
DATA_URLS: dict[str, str] = {
    "results": "https://raw.githubusercontent.com/martj42/international_results/master/results.csv",
    "shootouts": "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv",
    "goalscorers": "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv",
}

DATASET_TEAM_BY_LOWER: dict[str, str] = {team_name.lower(): team_name for team_name in FIFA_TO_DATASET_TEAM.values()}


def resolve_team_name(value: str) -> str:
    """Resolve a FIFA code or a dataset team name to the dataset spelling."""
    token = value.strip()
    if not token:
        raise ValueError("Team name cannot be empty.")
    fifa_match = FIFA_TO_DATASET_TEAM.get(token.upper())
    if fifa_match is not None:
        return fifa_match
    return DATASET_TEAM_BY_LOWER.get(token.lower(), token)


FIXTURE_HOME_TEAM: str = resolve_team_name(FIXTURE_HOME_TEAM_INPUT)
FIXTURE_AWAY_TEAM: str = resolve_team_name(FIXTURE_AWAY_TEAM_INPUT)


def load_csv_from_url(url: str) -> pl.DataFrame:
    """Load a CSV file from a URL into Polars.

    Args:
        url: Source URL pointing to a CSV file.

    Returns:
        A Polars DataFrame containing the parsed CSV data.
    """
    with urlopen(url) as response:
        payload = response.read()
    return pl.read_csv(BytesIO(payload), try_parse_dates=True, null_values=["NA"])


@dataclass(frozen=True)
class DixonColesFit:
    """Container for the fitted Dixon-Coles layer.

    Attributes:
        teams: Ordered team names used in the fit.
        attack: Team attack strengths on a centered scale.
        defense: Team defensive weakness values on a centered scale.
        home_advantage: Estimated home advantage on the log-goal scale.
        rho: Low-score dependency parameter.
    """

    teams: list[str]
    attack: np.ndarray
    defense: np.ndarray
    home_advantage: float
    rho: float


@dataclass(frozen=True)
class BayesianGoalModel:
    """Posterior samples for the Bayesian Poisson model.

    Attributes:
        feature_names: Names of the design-matrix columns.
        samples_home: Posterior draws for the home-goals coefficients.
        samples_away: Posterior draws for the away-goals coefficients.
    """

    feature_names: list[str]
    samples_home: np.ndarray
    samples_away: np.ndarray


@dataclass(frozen=True)
class XGBoostGoalModel:
    """Wrapper for the fitted XGBoost Poisson models.

    Attributes:
        feature_names: Names of the design-matrix columns.
        home_model: XGBRegressor used for home-goals prediction.
        away_model: XGBRegressor used for away-goals prediction.
    """

    feature_names: list[str]
    home_model: XGBRegressor
    away_model: XGBRegressor


def competition_weight(tournament: str) -> float:
    """Assign an importance weight to a competition.

    Args:
        tournament: Tournament name from the source dataset.

    Returns:
        A positive scalar weight where more important competitions receive larger values.
    """
    label = tournament.lower()
    if "world cup" in label:
        return 4.0
    if "copa america" in label or "euro" in label or "gold cup" in label or "africa cup" in label:
        return 3.0
    if "nations league" in label or "confederations" in label:
        return 2.5
    if "qual" in label or "oceanian nations cup" in label:
        return 1.8
    if "friendly" in label:
        return 1.0
    return 1.4


def time_decay_weight(match_date: np.datetime64, reference_date: np.datetime64, half_life_days: float = 365.25 * 3.0) -> float:
    """Compute an exponential recency weight.

    Args:
        match_date: Match date.
        reference_date: Reference date for the decay curve.
        half_life_days: Half-life of the decay in days.

    Returns:
        A recency weight in the interval (0, 1].
    """
    delta_days = float((reference_date - match_date) / np.timedelta64(1, "D"))
    return float(np.exp(-np.log(2.0) * delta_days / half_life_days))


def log_poisson_pmf(y: np.ndarray, rate: np.ndarray) -> np.ndarray:
    """Compute the Poisson log PMF with numpy arrays.

    Args:
        y: Observed counts.
        rate: Poisson mean values.

    Returns:
        Elementwise log probability values.
    """
    return y * np.log(rate) - rate - gammaln(y + 1.0)


def scoreline_matrix(lam_home: float, lam_away: float, max_goals: int = 10, rho: float = 0.0) -> np.ndarray:
    """Build a score probability matrix from independent Poisson rates.

    Args:
        lam_home: Expected home goals.
        lam_away: Expected away goals.
        max_goals: Maximum score to include on each axis.
        rho: Dixon-Coles low-score adjustment parameter.

    Returns:
        A square matrix of joint score probabilities.
    """
    home_grid = np.arange(max_goals + 1, dtype=float)
    away_grid = np.arange(max_goals + 1, dtype=float)
    home_pmf = np.exp(log_poisson_pmf(home_grid, np.full_like(home_grid, lam_home, dtype=float)))
    away_pmf = np.exp(log_poisson_pmf(away_grid, np.full_like(away_grid, lam_away, dtype=float)))
    matrix = np.outer(home_pmf, away_pmf)

    if max_goals >= 1:
        tau = np.ones_like(matrix)
        tau[0, 0] = 1.0 - (lam_home * lam_away * rho)
        tau[0, 1] = 1.0 + (lam_home * rho)
        tau[1, 0] = 1.0 + (lam_away * rho)
        tau[1, 1] = 1.0 - rho
        matrix = matrix * np.clip(tau, 0.05, None)

    return matrix / matrix.sum()

In [ ]:
RESULTS: pl.DataFrame = load_csv_from_url(DATA_URLS["results"]).with_columns(
    pl.col("date").cast(pl.Date)
)
SHOOTOUTS: pl.DataFrame = load_csv_from_url(DATA_URLS["shootouts"]).with_columns(
    pl.col("date").cast(pl.Date)
)
GOALSCORERS: pl.DataFrame = load_csv_from_url(DATA_URLS["goalscorers"]).with_columns(
    pl.col("date").cast(pl.Date)
)

MATCHES: pl.DataFrame = (
    RESULTS.filter(pl.col("date") >= pl.date(2018, 1, 1))
    .with_columns(
        pl.col("tournament").map_elements(competition_weight, return_dtype=pl.Float64).alias("competition_weight")
    )
    .sort("date")
)

REFERENCE_DATE = MATCHES.select(pl.col("date").max()).item()
MATCHES = MATCHES.with_columns(
    pl.col("date")
    .map_elements(
        lambda value: time_decay_weight(np.datetime64(value), np.datetime64(REFERENCE_DATE)),
        return_dtype=pl.Float64,
    )
    .alias("recency_weight")
).with_columns(
    (pl.col("competition_weight") * pl.col("recency_weight")).alias("sample_weight")
)

TRAIN_MATCHES: pl.DataFrame = MATCHES.filter(pl.col("date") < pl.date(2024, 1, 1))
TEST_MATCHES: pl.DataFrame = MATCHES.filter(pl.col("date") >= pl.date(2024, 1, 1))
TEAM_NAMES: list[str] = sorted(
    set(TRAIN_MATCHES.get_column("home_team").to_list()) | set(TRAIN_MATCHES.get_column("away_team").to_list())
)
TEAM_TO_INDEX: dict[str, int] = {team: index for index, team in enumerate(TEAM_NAMES)}
INDEX_TO_TEAM: dict[int, str] = {index: team for team, index in TEAM_TO_INDEX.items()}

print(f"Matches loaded: {MATCHES.height:,}")
print(f"Train / test split: {TRAIN_MATCHES.height:,} / {TEST_MATCHES.height:,}")
print(f"Teams in training window: {len(TEAM_NAMES)}")
print(MATCHES.select([
    pl.col("date").min().alias("first_date"),
    pl.col("date").max().alias("last_date"),
    pl.col("competition_weight").mean().alias("avg_competition_weight"),
    pl.col("sample_weight").mean().alias("avg_sample_weight"),
]).to_dicts()[0])

In [ ]:
def _unpack_dc_params(params: np.ndarray, team_count: int) -> tuple[np.ndarray, np.ndarray, float, float]:
    """Unpack the flattened Dixon-Coles parameter vector.

    Args:
        params: Flattened optimization vector.
        team_count: Number of teams included in the fit.

    Returns:
        A tuple with attack, defense, home advantage, and rho.
    """
    raw_count = team_count - 1
    attack_raw = params[:raw_count]
    defense_raw = params[raw_count : 2 * raw_count]
    home_advantage = float(params[2 * raw_count])
    rho_raw = float(params[2 * raw_count + 1])

    attack_full = np.concatenate([attack_raw, [-attack_raw.sum()]])
    defense_full = np.concatenate([defense_raw, [-defense_raw.sum()]])
    rho = 0.2 * np.tanh(rho_raw)
    return attack_full, defense_full, home_advantage, rho


def _dc_objective_and_gradient(
    params: np.ndarray,
    home_index: np.ndarray,
    away_index: np.ndarray,
    home_goals: np.ndarray,
    away_goals: np.ndarray,
    sample_weight: np.ndarray,
    team_count: int,
    prior_scale: float = 1.5,
) -> tuple[float, np.ndarray]:
    """Compute the negative weighted log posterior and its gradient.

    Args:
        params: Flattened parameter vector.
        home_index: Home-team indices for each match.
        away_index: Away-team indices for each match.
        home_goals: Observed home goals.
        away_goals: Observed away goals.
        sample_weight: Per-match weights combining time decay and competition importance.
        team_count: Number of teams in the training window.
        prior_scale: Normal prior scale for regularization.

    Returns:
        A tuple with the objective value and gradient vector.
    """
    attack_full, defense_full, home_advantage, rho = _unpack_dc_params(params, team_count)
    rho_raw = float(np.arctanh(np.clip(rho / 0.2, -0.999999, 0.999999)))

    home_attack = attack_full[home_index]
    away_attack = attack_full[away_index]
    home_defense = defense_full[home_index]
    away_defense = defense_full[away_index]

    eta_home = home_advantage + home_attack + away_defense
    eta_away = away_attack + home_defense
    lambda_home = np.exp(np.clip(eta_home, -10.0, 10.0))
    lambda_away = np.exp(np.clip(eta_away, -10.0, 10.0))

    base_loglik = sample_weight * (
        home_goals * eta_home
        - lambda_home
        + away_goals * eta_away
        - lambda_away
        - gammaln(home_goals + 1.0)
        - gammaln(away_goals + 1.0)
    )

    tau = np.ones_like(base_loglik)
    grad_eta_home = sample_weight * (home_goals - lambda_home)
    grad_eta_away = sample_weight * (away_goals - lambda_away)
    grad_rho = np.zeros_like(base_loglik)

    mask_00 = (home_goals == 0) & (away_goals == 0)
    if np.any(mask_00):
        tau_00 = np.clip(1.0 - (lambda_home[mask_00] * lambda_away[mask_00] * rho), 1e-9, None)
        grad_eta_home[mask_00] += sample_weight[mask_00] * (-(lambda_home[mask_00] * lambda_away[mask_00] * rho) / tau_00)
        grad_eta_away[mask_00] += sample_weight[mask_00] * (-(lambda_home[mask_00] * lambda_away[mask_00] * rho) / tau_00)
        grad_rho[mask_00] = sample_weight[mask_00] * (-(lambda_home[mask_00] * lambda_away[mask_00]) / tau_00)
        tau[mask_00] = tau_00

    mask_01 = (home_goals == 0) & (away_goals == 1)
    if np.any(mask_01):
        tau_01 = np.clip(1.0 + (lambda_home[mask_01] * rho), 1e-9, None)
        grad_eta_home[mask_01] += sample_weight[mask_01] * ((lambda_home[mask_01] * rho) / tau_01)
        grad_rho[mask_01] = sample_weight[mask_01] * (lambda_home[mask_01] / tau_01)
        tau[mask_01] = tau_01

    mask_10 = (home_goals == 1) & (away_goals == 0)
    if np.any(mask_10):
        tau_10 = np.clip(1.0 + (lambda_away[mask_10] * rho), 1e-9, None)
        grad_eta_away[mask_10] += sample_weight[mask_10] * ((lambda_away[mask_10] * rho) / tau_10)
        grad_rho[mask_10] = sample_weight[mask_10] * (lambda_away[mask_10] / tau_10)
        tau[mask_10] = tau_10

    mask_11 = (home_goals == 1) & (away_goals == 1)
    if np.any(mask_11):
        tau_11 = np.clip(1.0 - rho, 1e-9, None)
        grad_rho[mask_11] = sample_weight[mask_11] * (-1.0 / tau_11)
        tau[mask_11] = tau_11

    log_posterior = float(np.sum(base_loglik + sample_weight * np.log(tau)))
    regularizer = 0.5 * np.sum((params / prior_scale) ** 2)
    objective = -(log_posterior - regularizer)

    grad_attack_full = np.bincount(home_index, weights=grad_eta_home, minlength=team_count) + np.bincount(
        away_index, weights=grad_eta_away, minlength=team_count
    )
    grad_defense_full = np.bincount(home_index, weights=grad_eta_away, minlength=team_count) + np.bincount(
        away_index, weights=grad_eta_home, minlength=team_count
    )

    raw_count = team_count - 1
    grad_attack_raw = grad_attack_full[:raw_count] - grad_attack_full[raw_count]
    grad_defense_raw = grad_defense_full[:raw_count] - grad_defense_full[raw_count]
    grad_home_advantage = float(np.sum(grad_eta_home))
    grad_rho_raw = float(np.sum(grad_rho) * (0.2 * (1.0 - np.tanh(rho_raw) ** 2)))

    gradient = -np.concatenate(
        [
            grad_attack_raw - params[:raw_count] / (prior_scale**2),
            grad_defense_raw - params[raw_count : 2 * raw_count] / (prior_scale**2),
            np.array([grad_home_advantage - params[2 * raw_count] / (prior_scale**2)]),
            np.array([grad_rho_raw - params[2 * raw_count + 1] / (prior_scale**2)]),
        ]
    )
    return objective, gradient


def fit_dixon_coles(train_matches: pl.DataFrame, team_names: list[str]) -> DixonColesFit:
    """Fit a weighted Dixon-Coles model.

    Args:
        train_matches: Training matches from the 2018+ window.
        team_names: Ordered list of team names used for the model.

    Returns:
        A fitted DixonColesFit object with centered team strengths.
    """
    home_index = train_matches.get_column("home_team").replace_strict(team_names, list(range(len(team_names)))).to_numpy()
    away_index = train_matches.get_column("away_team").replace_strict(team_names, list(range(len(team_names)))).to_numpy()
    home_goals = train_matches.get_column("home_score").to_numpy().astype(float)
    away_goals = train_matches.get_column("away_score").to_numpy().astype(float)
    sample_weight = train_matches.get_column("sample_weight").to_numpy().astype(float)
    sample_weight = sample_weight / np.mean(sample_weight)

    team_count = len(team_names)
    raw_count = team_count - 1
    initial_params = np.zeros(2 * raw_count + 2, dtype=float)
    initial_params[-2] = np.log(np.mean(home_goals + 1.0) / np.mean(away_goals + 1.0))

    result = minimize(
        lambda vector: _dc_objective_and_gradient(
            vector,
            home_index,
            away_index,
            home_goals,
            away_goals,
            sample_weight,
            team_count,
        ),
        initial_params,
        method="L-BFGS-B",
        jac=True,
        options={"maxiter": 200, "ftol": 1e-8},
    )

    attack_full, defense_full, home_advantage, rho = _unpack_dc_params(result.x, team_count)
    attack_full = attack_full - np.mean(attack_full)
    defense_full = defense_full - np.mean(defense_full)

    print(f"Dixon-Coles optimization converged: {result.success} ({result.message})")
    print(f"Final weighted negative log posterior: {result.fun:,.2f}")

    return DixonColesFit(
        teams=team_names,
        attack=attack_full,
        defense=defense_full,
        home_advantage=float(home_advantage),
        rho=float(rho),
    )


def build_team_strength_frame(dc_fit: DixonColesFit) -> pl.DataFrame:
    """Create a readable team-strength table from the fitted model.

    Args:
        dc_fit: Fitted Dixon-Coles model.

    Returns:
        A Polars DataFrame with attack and defensive strength summaries.
    """
    return pl.DataFrame(
        {
            "team": dc_fit.teams,
            "attack_strength": dc_fit.attack,
            "defensive_weakness": dc_fit.defense,
            "defensive_power": -dc_fit.defense,
        }
    ).sort("attack_strength", descending=True)


DC_FIT: DixonColesFit = fit_dixon_coles(TRAIN_MATCHES, TEAM_NAMES)
TEAM_STRENGTHS: pl.DataFrame = build_team_strength_frame(DC_FIT)

print(TEAM_STRENGTHS.head(10))

In [ ]:
FEATURE_COLUMNS: list[str] = [
    "home_attack",
    "away_attack",
    "home_defensive_power",
    "away_defensive_power",
    "attack_diff",
    "defense_diff",
    "home_advantage_flag",
]


def add_strength_features(matches: pl.DataFrame, team_strengths: pl.DataFrame) -> pl.DataFrame:
    """Join fitted team strengths to a match table.

    Args:
        matches: Match-level table with home and away teams.
        team_strengths: Team-level attack and defense table.

    Returns:
        A match table augmented with strength-derived features.
    """
    home_strengths = team_strengths.select(
        pl.col("team").alias("home_team"),
        pl.col("attack_strength").alias("home_attack"),
        pl.col("defensive_power").alias("home_defensive_power"),
    )
    away_strengths = team_strengths.select(
        pl.col("team").alias("away_team"),
        pl.col("attack_strength").alias("away_attack"),
        pl.col("defensive_power").alias("away_defensive_power"),
    )

    return (
        matches.join(home_strengths, on="home_team", how="left")
        .join(away_strengths, on="away_team", how="left")
        .with_columns(
            (pl.col("home_attack") - pl.col("away_attack")).alias("attack_diff"),
            (pl.col("home_defensive_power") - pl.col("away_defensive_power")).alias("defense_diff"),
            (1.0 - pl.col("neutral").cast(pl.Float64)).alias("home_advantage_flag"),
        )
    )


def to_matrix(frame: pl.DataFrame, columns: list[str]) -> np.ndarray:
    """Convert a Polars table into a 2D numpy feature matrix.

    Args:
        frame: Input table.
        columns: Ordered column names to extract.

    Returns:
        A dense 2D numpy array.
    """
    return np.column_stack([frame.get_column(column).to_numpy().astype(float) for column in columns])


TRAIN_FEATURES: pl.DataFrame = add_strength_features(TRAIN_MATCHES, TEAM_STRENGTHS)
TEST_FEATURES: pl.DataFrame = add_strength_features(TEST_MATCHES, TEAM_STRENGTHS)

X_TRAIN: np.ndarray = to_matrix(TRAIN_FEATURES, FEATURE_COLUMNS)
X_TEST: np.ndarray = to_matrix(TEST_FEATURES, FEATURE_COLUMNS)
Y_TRAIN_HOME: np.ndarray = TRAIN_FEATURES.get_column("home_score").to_numpy().astype(float)
Y_TRAIN_AWAY: np.ndarray = TRAIN_FEATURES.get_column("away_score").to_numpy().astype(float)
Y_TEST_HOME: np.ndarray = TEST_FEATURES.get_column("home_score").to_numpy().astype(float)
Y_TEST_AWAY: np.ndarray = TEST_FEATURES.get_column("away_score").to_numpy().astype(float)
SAMPLE_WEIGHT_TRAIN: np.ndarray = TRAIN_FEATURES.get_column("sample_weight").to_numpy().astype(float)
SAMPLE_WEIGHT_TRAIN = SAMPLE_WEIGHT_TRAIN / np.mean(SAMPLE_WEIGHT_TRAIN)

print(TRAIN_FEATURES.select(FEATURE_COLUMNS).head(5))
print(f"Feature matrix shape: {X_TRAIN.shape}")

## Bayesian and XGBoost models

The downstream models share the same team-strength features but learn different parameterizations.

- The Bayesian model samples coefficients with MCMC, giving posterior uncertainty for each fixture.
- The XGBoost model learns a nonlinear Poisson mapping that can capture feature interactions.

In [ ]:
def standardize_matrix(train_matrix: np.ndarray, test_matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Standardize a feature matrix with training-set statistics.

    Args:
        train_matrix: Training design matrix.
        test_matrix: Test design matrix.

    Returns:
        A tuple with standardized train and test matrices, plus means and scales.
    """
    means = train_matrix.mean(axis=0)
    scales = train_matrix.std(axis=0)
    scales = np.where(scales == 0.0, 1.0, scales)
    return (train_matrix - means) / scales, (test_matrix - means) / scales, means, scales


def poisson_log_posterior(beta: np.ndarray, design_matrix: np.ndarray, targets: np.ndarray, sample_weight: np.ndarray, prior_scale: float = 2.5) -> float:
    """Evaluate the log posterior for a weighted Poisson regression.

    Args:
        beta: Regression coefficients including the intercept.
        design_matrix: Standardized feature matrix with an intercept column.
        targets: Goal counts for one side of the fixture.
        sample_weight: Per-match weights.
        prior_scale: Normal prior scale for coefficients.

    Returns:
        The scalar log posterior value.
    """
    eta = design_matrix @ beta
    rate = np.exp(np.clip(eta, -10.0, 10.0))
    log_likelihood = np.sum(sample_weight * (targets * eta - rate - gammaln(targets + 1.0)))
    log_prior = -0.5 * np.sum((beta / prior_scale) ** 2)
    return float(log_likelihood + log_prior)


def metropolis_hastings_poisson(
    design_matrix: np.ndarray,
    targets: np.ndarray,
    sample_weight: np.ndarray,
    n_steps: int = 2500,
    burn_in: int = 700,
    thin: int = 3,
    prior_scale: float = 2.5,
) -> np.ndarray:
    """Sample a weighted Poisson regression with random-walk Metropolis.

    Args:
        design_matrix: Standardized feature matrix with an intercept column.
        targets: Goal counts for one side of the fixture.
        sample_weight: Per-match weights.
        n_steps: Total MCMC iterations.
        burn_in: Number of initial iterations to discard.
        thin: Keep one draw every `thin` iterations after burn-in.
        prior_scale: Normal prior scale for coefficients.

    Returns:
        An array of posterior coefficient samples.
    """
    beta_current = np.zeros(design_matrix.shape[1], dtype=float)
    log_post_current = poisson_log_posterior(beta_current, design_matrix, targets, sample_weight, prior_scale=prior_scale)
    proposal_scale = np.full(beta_current.shape, 0.04, dtype=float)
    draws: list[np.ndarray] = []
    accepted = 0
    window_accepted = 0

    for step in range(n_steps):
        proposal = beta_current + RNG.normal(scale=proposal_scale, size=beta_current.shape)
        log_post_proposal = poisson_log_posterior(proposal, design_matrix, targets, sample_weight, prior_scale=prior_scale)
        log_acceptance = log_post_proposal - log_post_current

        if np.log(RNG.uniform()) < log_acceptance:
            beta_current = proposal
            log_post_current = log_post_proposal
            accepted += 1
            window_accepted += 1

        if step >= burn_in and (step - burn_in) % thin == 0:
            draws.append(beta_current.copy())

        if step > 0 and (step + 1) % 250 == 0:
            window_rate = window_accepted / 250.0
            if window_rate < 0.15:
                proposal_scale *= 0.75
            elif window_rate > 0.4:
                proposal_scale *= 1.2
            window_accepted = 0

    acceptance_rate = accepted / n_steps
    print(f"MCMC acceptance rate: {acceptance_rate:.3f}")
    return np.vstack(draws)


def fit_xgboost_goal_models(
    train_design_matrix: np.ndarray,
    train_home_targets: np.ndarray,
    train_away_targets: np.ndarray,
    sample_weight: np.ndarray,
) -> XGBoostGoalModel:
    """Fit paired Poisson XGBoost regressors for home and away goals.

    Args:
        train_design_matrix: Standardized feature matrix with an intercept column.
        train_home_targets: Home-goal counts.
        train_away_targets: Away-goal counts.
        sample_weight: Per-match weights.

    Returns:
        A wrapper containing the fitted XGBoost regressors.
    """
    xgb_params: dict[str, object] = {
        "objective": "count:poisson",
        "n_estimators": 350,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 1.0,
        "random_state": 42,
    }
    home_model = XGBRegressor(**xgb_params)
    away_model = XGBRegressor(**xgb_params)
    home_model.fit(train_design_matrix, train_home_targets, sample_weight=sample_weight)
    away_model.fit(train_design_matrix, train_away_targets, sample_weight=sample_weight)
    return XGBoostGoalModel(feature_names=["intercept"] + FEATURE_COLUMNS, home_model=home_model, away_model=away_model)


X_TRAIN_STD, X_TEST_STD, FEATURE_MEANS, FEATURE_SCALES = standardize_matrix(X_TRAIN, X_TEST)
X_TRAIN_MODEL: np.ndarray = np.column_stack([np.ones(X_TRAIN_STD.shape[0]), X_TRAIN_STD])
X_TEST_MODEL: np.ndarray = np.column_stack([np.ones(X_TEST_STD.shape[0]), X_TEST_STD])

BAYES_HOME_SAMPLES: np.ndarray = metropolis_hastings_poisson(X_TRAIN_MODEL, Y_TRAIN_HOME, SAMPLE_WEIGHT_TRAIN)
BAYES_AWAY_SAMPLES: np.ndarray = metropolis_hastings_poisson(X_TRAIN_MODEL, Y_TRAIN_AWAY, SAMPLE_WEIGHT_TRAIN)
BAYESIAN_MODEL: BayesianGoalModel = BayesianGoalModel(
    feature_names=["intercept"] + FEATURE_COLUMNS,
    samples_home=BAYES_HOME_SAMPLES,
    samples_away=BAYES_AWAY_SAMPLES,
)
XGB_MODEL: XGBoostGoalModel = fit_xgboost_goal_models(X_TRAIN_MODEL, Y_TRAIN_HOME, Y_TRAIN_AWAY, SAMPLE_WEIGHT_TRAIN)

print(f"Bayesian samples retained: {BAYES_HOME_SAMPLES.shape[0]}")
print(f"Bayesian feature count: {len(BAYESIAN_MODEL.feature_names)}")

## Demo fixture prediction

The demo fixture is Argentina vs Austria on 22 June. The notebook keeps the venue flag editable so you can switch between neutral and home/away assumptions without changing the model pipeline.

In [ ]:
def build_fixture_features(home_team: str, away_team: str, neutral: bool) -> pl.DataFrame:
    """Create a single-match feature frame for prediction.

    Args:
        home_team: Home team name.
        away_team: Away team name.
        neutral: Whether the fixture is on neutral ground.

    Returns:
        A one-row Polars DataFrame ready for feature transformation.
    """
    base_fixture = pl.DataFrame(
        {
            "date": ["2024-06-22"],
            "home_team": [home_team],
            "away_team": [away_team],
            "home_score": [0],
            "away_score": [0],
            "tournament": ["Friendly"],
            "city": [""],
            "country": [""],
            "neutral": [neutral],
            "competition_weight": [competition_weight("Friendly")],
            "recency_weight": [1.0],
            "sample_weight": [1.0],
        }
    ).with_columns(pl.col("date").cast(pl.Date))
    return add_strength_features(base_fixture, TEAM_STRENGTHS)


def model_row(matrix_frame: pl.DataFrame) -> np.ndarray:
    """Transform a one-row feature frame into the model matrix.

    Args:
        matrix_frame: Single-row frame with the engineered features.

    Returns:
        A 2D numpy array with intercept and standardized features.
    """
    raw_row = to_matrix(matrix_frame, FEATURE_COLUMNS)
    standardized_row = (raw_row - FEATURE_MEANS) / FEATURE_SCALES
    return np.column_stack([np.ones(standardized_row.shape[0]), standardized_row])


def posterior_rate_samples(design_row: np.ndarray, coefficient_samples: np.ndarray) -> np.ndarray:
    """Convert posterior coefficient draws into Poisson mean draws.

    Args:
        design_row: One-row design matrix including intercept.
        coefficient_samples: Posterior samples for one target.

    Returns:
        A one-dimensional array of Poisson rate samples.
    """
    return np.exp(np.clip(design_row @ coefficient_samples.T, -10.0, 10.0)).ravel()


def aggregate_score_matrices(home_rates: np.ndarray, away_rates: np.ndarray, max_goals: int = 8) -> np.ndarray:
    """Average score matrices across posterior draws.

    Args:
        home_rates: Posterior samples of the home-goal mean.
        away_rates: Posterior samples of the away-goal mean.
        max_goals: Maximum score to include on each axis.

    Returns:
        A normalized probability matrix.
    """
    matrix_sum = np.zeros((max_goals + 1, max_goals + 1), dtype=float)
    draw_count = min(len(home_rates), len(away_rates))
    chosen = np.linspace(0, draw_count - 1, num=min(200, draw_count), dtype=int)
    for index in chosen:
        matrix_sum += scoreline_matrix(float(home_rates[index]), float(away_rates[index]), max_goals=max_goals, rho=DC_FIT.rho)
    return matrix_sum / matrix_sum.sum()


def outcome_probabilities(matrix: np.ndarray) -> tuple[float, float, float]:
    """Collapse a score matrix into win/draw probabilities.

    Args:
        matrix: Joint score probability matrix.

    Returns:
        The home-win, draw, and away-win probabilities.
    """
    home_win = float(np.tril(matrix, k=-1).sum())
    draw = float(np.trace(matrix))
    away_win = float(np.triu(matrix, k=1).sum())
    return home_win, draw, away_win


def top_scorelines(matrix: np.ndarray, k: int = 10) -> pl.DataFrame:
    """Extract the most probable scorelines from a matrix.

    Args:
        matrix: Joint score probability matrix.
        k: Number of scorelines to return.

    Returns:
        A Polars DataFrame sorted by probability.
    """
    score_rows, score_cols = np.indices(matrix.shape)
    flattened = pl.DataFrame(
        {
            "home_goals": score_rows.ravel(),
            "away_goals": score_cols.ravel(),
            "probability": matrix.ravel(),
        }
    )
    return (
        flattened.with_columns(pl.format("{}-{}", pl.col("home_goals"), pl.col("away_goals")).alias("scoreline"))
        .sort("probability", descending=True)
        .head(k)
    )

# Prediction

In [ ]:
FIXTURE_FEATURES: pl.DataFrame = build_fixture_features(FIXTURE_HOME_TEAM, FIXTURE_AWAY_TEAM, FIXTURE_NEUTRAL)
FIXTURE_MODEL_ROW: np.ndarray = model_row(FIXTURE_FEATURES)

FIXTURE_BAYES_HOME_RATES: np.ndarray = posterior_rate_samples(FIXTURE_MODEL_ROW, BAYES_HOME_SAMPLES)
FIXTURE_BAYES_AWAY_RATES: np.ndarray = posterior_rate_samples(FIXTURE_MODEL_ROW, BAYES_AWAY_SAMPLES)
FIXTURE_BAYES_MATRIX: np.ndarray = aggregate_score_matrices(FIXTURE_BAYES_HOME_RATES, FIXTURE_BAYES_AWAY_RATES, max_goals=8)
FIXTURE_XGB_HOME_RATE: float = float(XGB_MODEL.home_model.predict(FIXTURE_MODEL_ROW)[0])
FIXTURE_XGB_AWAY_RATE: float = float(XGB_MODEL.away_model.predict(FIXTURE_MODEL_ROW)[0])
FIXTURE_XGB_MATRIX: np.ndarray = scoreline_matrix(FIXTURE_XGB_HOME_RATE, FIXTURE_XGB_AWAY_RATE, max_goals=8, rho=DC_FIT.rho)
FIXTURE_ENSEMBLE_MATRIX: np.ndarray = 0.5 * FIXTURE_BAYES_MATRIX + 0.5 * FIXTURE_XGB_MATRIX
FIXTURE_ENSEMBLE_MATRIX = FIXTURE_ENSEMBLE_MATRIX / FIXTURE_ENSEMBLE_MATRIX.sum()


def format_percent(value: float, significant_figures: int = 3) -> str:
    """Format a probability as a percentage with a target number of significant figures.

    Args:
        value: Probability in the range [0, 1].
        significant_figures: Desired significant figures for display.

    Returns:
        A percentage string such as 67.4%.
    """
    percent_value = value * 100.0
    if np.isclose(percent_value, 0.0):
        return "0%"
    decimals = max(significant_figures - int(np.floor(np.log10(abs(percent_value)))) - 1, 0)
    return f"{percent_value:.{decimals}f}%"


fixture_home_win, fixture_draw, fixture_away_win = outcome_probabilities(FIXTURE_ENSEMBLE_MATRIX)
fixture_top_scores = top_scorelines(FIXTURE_ENSEMBLE_MATRIX, k=10)
fixture_modal_index = np.unravel_index(np.argmax(FIXTURE_ENSEMBLE_MATRIX), FIXTURE_ENSEMBLE_MATRIX.shape)

print(f"Fixture: {FIXTURE_HOME_TEAM} vs {FIXTURE_AWAY_TEAM}")
print(f"Posterior mean goals, Bayesian model: {FIXTURE_BAYES_HOME_RATES.mean():.2f} - {FIXTURE_BAYES_AWAY_RATES.mean():.2f}")
print(f"Point estimate goals, XGBoost model: {FIXTURE_XGB_HOME_RATE:.2f} - {FIXTURE_XGB_AWAY_RATE:.2f}")
print(f"Ensemble modal score: {fixture_modal_index[0]}-{fixture_modal_index[1]}")
print(
    "Win / draw / win probabilities: "
    f"{format_percent(fixture_home_win)} / {format_percent(fixture_draw)} / {format_percent(fixture_away_win)}"
)
print(fixture_top_scores)

## Probability visualizations

The next cell renders the ensemble prediction as a scoreline heatmap, a win/draw probability chart, and a top-10 score chart.

In [ ]:
score_labels: list[str] = [f"{home}-{away}" for home in range(FIXTURE_ENSEMBLE_MATRIX.shape[0]) for away in range(FIXTURE_ENSEMBLE_MATRIX.shape[1])]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    FIXTURE_ENSEMBLE_MATRIX,
    ax=ax,
    cmap="mako",
    square=True,
    cbar_kws={"label": f"{PROBABILITY_AXIS_LABEL}"},
)
ax.set_title(f"{FIXTURE_HOME_TEAM} vs {FIXTURE_AWAY_TEAM} score probability heatmap")
ax.set_xlabel(f"Away ({FIXTURE_AWAY_TEAM}) goals")
ax.set_ylabel(f"Home ({FIXTURE_HOME_TEAM}) goals")
ax.set_xticks(np.arange(FIXTURE_ENSEMBLE_MATRIX.shape[1]) + 0.5)
ax.set_xticklabels(range(FIXTURE_ENSEMBLE_MATRIX.shape[1]))
ax.set_yticks(np.arange(FIXTURE_ENSEMBLE_MATRIX.shape[0]) + 0.5)
ax.set_yticklabels(range(FIXTURE_ENSEMBLE_MATRIX.shape[0]), rotation=0)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

probability_summary = pl.DataFrame(
    {
        "outcome": [f"{FIXTURE_HOME_TEAM} win", "Draw", f"{FIXTURE_AWAY_TEAM} win"],
        "probability": [fixture_home_win, fixture_draw, fixture_away_win],
    }
)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2a9d8f", "#e9c46a", "#e76f51"]
ax.bar(probability_summary.get_column("outcome").to_list(), probability_summary.get_column("probability").to_list(), color=colors)
ax.set_ylim(0, 1)
ax.set_ylabel(f"{PROBABILITY_AXIS_LABEL}")
ax.set_title(f"Match outcome probabilities: {FIXTURE_HOME_TEAM} vs {FIXTURE_AWAY_TEAM}")
for index, value in enumerate(probability_summary.get_column("probability").to_list()):
    ax.text(index, value + 0.02, format_percent(value), ha="center", va="bottom")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ranked_scores = fixture_top_scores.with_columns(
    pl.format(f"{FIXTURE_HOME_TEAM_INPUT} {{}} {FIXTURE_AWAY_TEAM_INPUT}", pl.col("scoreline")).alias("display_scoreline")
).sort("probability")
ax.barh(ranked_scores.get_column("display_scoreline").to_list(), ranked_scores.get_column("probability").to_list(), color="#457b9d")
ax.set_xlabel(f"{PROBABILITY_AXIS_LABEL}")
ax.set_title(f"Top 10 most probable scorelines: {FIXTURE_HOME_TEAM} vs {FIXTURE_AWAY_TEAM}")
for row_index, probability in enumerate(ranked_scores.get_column("probability").to_list()):
    ax.text(probability + 0.002, row_index, format_percent(probability), va="center")
plt.tight_layout()
plt.show()

## Holdout evaluation

The next cell measures how the Bayesian model, XGBoost, and their ensemble perform on the 2024 holdout window using goal error, outcome accuracy, exact score hit rate, and multiclass log loss.

In [ ]:
def predict_rate_vectors(design_matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Predict home and away goal rates for each model.

    Args:
        design_matrix: Standardized feature matrix with an intercept column.

    Returns:
        A tuple containing Bayesian, XGBoost, and ensemble goal-rate vectors.
    """
    bayes_home = np.exp(np.clip(design_matrix @ BAYES_HOME_SAMPLES.T, -10.0, 10.0)).mean(axis=1)
    bayes_away = np.exp(np.clip(design_matrix @ BAYES_AWAY_SAMPLES.T, -10.0, 10.0)).mean(axis=1)
    xgb_home = XGB_MODEL.home_model.predict(design_matrix)
    xgb_away = XGB_MODEL.away_model.predict(design_matrix)
    ensemble_home = 0.5 * (bayes_home + xgb_home)
    ensemble_away = 0.5 * (bayes_away + xgb_away)
    return bayes_home, bayes_away, xgb_home, xgb_away, ensemble_home, ensemble_away


def evaluate_model_predictions(
    home_rates: np.ndarray,
    away_rates: np.ndarray,
    actual_home: np.ndarray,
    actual_away: np.ndarray,
    max_goals: int = 12,
) -> dict[str, float]:
    """Evaluate a score model on a holdout set.

    Args:
        home_rates: Predicted home-goal means.
        away_rates: Predicted away-goal means.
        actual_home: Observed home goals.
        actual_away: Observed away goals.
        max_goals: Maximum score to include when forming the probability matrix.

    Returns:
        A dictionary of scalar evaluation metrics.
    """
    predicted_home = np.asarray(home_rates, dtype=float)
    predicted_away = np.asarray(away_rates, dtype=float)
    actual_home = np.asarray(actual_home, dtype=float)
    actual_away = np.asarray(actual_away, dtype=float)
    valid_mask = (
        ~np.isnan(predicted_home)
        & ~np.isnan(predicted_away)
        & ~np.isnan(actual_home)
        & ~np.isnan(actual_away)
    )
    predicted_home = predicted_home[valid_mask]
    predicted_away = predicted_away[valid_mask]
    actual_home = actual_home[valid_mask]
    actual_away = actual_away[valid_mask]
    exact_score_hits = 0
    outcome_truth: list[int] = []
    outcome_probabilities_predicted: list[list[float]] = []
    score_nll = 0.0

    for home_rate, away_rate, observed_home, observed_away in zip(predicted_home, predicted_away, actual_home, actual_away):
        matrix = scoreline_matrix(float(home_rate), float(away_rate), max_goals=max_goals, rho=DC_FIT.rho)
        observed_home_int = int(observed_home)
        observed_away_int = int(observed_away)
        clipped_home = min(observed_home_int, max_goals)
        clipped_away = min(observed_away_int, max_goals)
        score_nll -= float(np.log(max(matrix[clipped_home, clipped_away], 1e-12)))
        exact_score_hits += int(np.argmax(matrix) == np.ravel_multi_index((clipped_home, clipped_away), matrix.shape))
        home_win, draw, away_win = outcome_probabilities(matrix)
        outcome_probabilities_predicted.append([home_win, draw, away_win])
        if observed_home_int > observed_away_int:
            outcome_truth.append(0)
        elif observed_home_int == observed_away_int:
            outcome_truth.append(1)
        else:
            outcome_truth.append(2)

    predicted_outcome = np.asarray(outcome_probabilities_predicted, dtype=float)
    outcome_labels = np.asarray(outcome_truth, dtype=int)
    outcome_class = np.argmax(predicted_outcome, axis=1)

    return {
        "mae_home": float(mean_absolute_error(actual_home, predicted_home)),
        "mae_away": float(mean_absolute_error(actual_away, predicted_away)),
        "mae_total": float(mean_absolute_error(actual_home + actual_away, predicted_home + predicted_away)),
        "outcome_accuracy": float(accuracy_score(outcome_labels, outcome_class)),
        "log_loss": float(log_loss(outcome_labels, predicted_outcome, labels=[0, 1, 2])),
        "exact_score_accuracy": float(exact_score_hits / len(predicted_home)),
        "mean_score_nll": float(score_nll / len(predicted_home)),
    }


BAYES_HOME_TEST, BAYES_AWAY_TEST, XGB_HOME_TEST, XGB_AWAY_TEST, ENSEMBLE_HOME_TEST, ENSEMBLE_AWAY_TEST = predict_rate_vectors(X_TEST_MODEL)

MODEL_METRICS: pl.DataFrame = pl.DataFrame(
    [
        {"model": "Bayesian MCMC", **evaluate_model_predictions(BAYES_HOME_TEST, BAYES_AWAY_TEST, Y_TEST_HOME, Y_TEST_AWAY)},
        {"model": "XGBoost", **evaluate_model_predictions(XGB_HOME_TEST, XGB_AWAY_TEST, Y_TEST_HOME, Y_TEST_AWAY)},
        {"model": "Ensemble", **evaluate_model_predictions(ENSEMBLE_HOME_TEST, ENSEMBLE_AWAY_TEST, Y_TEST_HOME, Y_TEST_AWAY)},
    ]
)

print(MODEL_METRICS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics_long = MODEL_METRICS.select([
    pl.col("model"),
    pl.col("outcome_accuracy"),
    pl.col("exact_score_accuracy"),
    pl.col("mae_total"),
])

axes[0].bar(metrics_long.get_column("model").to_list(), metrics_long.get_column("outcome_accuracy").to_list(), color=["#2a9d8f", "#457b9d", "#e76f51"])
axes[0].set_ylim(0, 1)
axes[0].set_title("Outcome accuracy")
axes[0].set_ylabel("Accuracy")

axes[1].bar(metrics_long.get_column("model").to_list(), metrics_long.get_column("mae_total").to_list(), color=["#2a9d8f", "#457b9d", "#e76f51"])
axes[1].set_title("Total-goals MAE")
axes[1].set_ylabel("MAE")

plt.tight_layout()
plt.show()